## Data Collection Notebook

In [1]:
import os

import pandas as pd

import sys
sys.path.append('../..')

from final_project.scripts.data_collection_utils import set_nulls, adjust_time_periods

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Reading CSV from EES - Explore Education Statistics

</h3>

    - https://explore-education-statistics.service.gov.uk/data-catalogue
    

</div>

In [2]:
rurality_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_rurality.csv"))
chars_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_characteristics.csv"))
retention_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_retention_by_region.csv"))
results_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_results_by_subject.csv"))
stem_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_stem_by_sex.csv"))

In [3]:
rurality_df = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/285687fd-7a12-4cbc-8a93-2fccfa0df053/csv")
chars_df = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/ca6f1e5a-35a5-4090-baf2-5611e43a7901/csv")
retention_df = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/1fece430-ce67-4027-ad93-068a4c8e1d5a/csv")
results_df = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/10aeadde-eece-4e45-a9b8-8b6c2fbd5124/csv",
                         low_memory=False)
stem_df = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/f46bb916-d67c-403f-8e01-586ae5c15e61/csv")

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out rurality_df

</h3>

    I will:
    - pull the important/ useful columns
    - deal with the custom null values "z", "x" and "c"
    - identify actual null values
    - give ordered categorical type to grade columns in order to perform numerical aggregation later (eg. min, max etc.)
    
</div>

In [4]:
rurality_df.groupby("rurality_name").agg({"number_of_students_potential": "sum"})

,number_of_students_potential
rurality_name,
Rural - Hamlet and isolated dwellings,57615
Rural - Town and fringe,88090
Rural - Village,29367
Unknown rurality,0
Urban - City and town,1432430
Urban - Major conurbation,1036029
Urban - Minor conurbation,97582


In [5]:
rurality_df = rurality_df[["time_period", "region_name", "rurality_name", "number_of_students_level3",
    "number_of_students_alev", "number_of_students_acad", "number_of_students_agen", "number_of_students_highest_entry_was_l2",
    "number_of_students_tlev", "number_of_students_technicalcertificate", "number_of_students_potential",
    "pc_achieving_3_astar_to_a_alev", "pc_achieving_atleast_two_alev", "aps_per_entry_grade_alev"]]

In [6]:
rurality_df = rurality_df[(rurality_df["rurality_name"] != "Unknown rurality")].reset_index(drop=True)
# 9 regions *5 time periods  *6 ruralities = 270 rows

rurality_df = rurality_df.map(set_nulls)

rurality_df["time_period"] = rurality_df["time_period"].apply(adjust_time_periods)

In [7]:
rurality_df["pc_achieving_alev"] = ((rurality_df["number_of_students_alev"] / rurality_df["number_of_students_potential"]) *100).round(3)

rurality_df.drop(columns=["number_of_students_level3", "number_of_students_acad", "number_of_students_agen",
                      "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate"],
                      inplace=True)

In [8]:
rurality_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   time_period                     270 non-null    object 
 1   region_name                     270 non-null    object 
 2   rurality_name                   270 non-null    object 
 3   number_of_students_alev         270 non-null    int64  
 4   number_of_students_potential    270 non-null    int64  
 5   pc_achieving_3_astar_to_a_alev  214 non-null    float64
 6   pc_achieving_atleast_two_alev   215 non-null    float64
 7   aps_per_entry_grade_alev        215 non-null    object 
 8   pc_achieving_alev               227 non-null    float64
dtypes: float64(3), int64(2), object(4)
memory usage: 19.1+ KB


In [9]:
with open(rurality_path, "w") as f:
    rurality_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out chars_df

</h3>

    I will:
    - pull the important/ useful columns
    - deal with the custom null values "z", "x" and "c"
    - identify actual null values
    - give ordered categorical type to grade columns in order to perform numerical aggregation later (eg. min, max etc.)
    
</div>

In [10]:
chars_df.groupby("characteristic_type").agg({"characteristic_value": "unique"})

,characteristic_value
characteristic_type,
All Students,[All state-funded students]
Disadvantage,"[Disadvantaged, Non-Disadvantaged, Unknown dis..."
Ethnicity,"[Black or Black British, Mixed Dual background..."
FSM,"[Eligible for FSM, Not eligible for FSM, Unkno..."
First Language,"[English language, Other than English language..."
Prior Attainment,"[Priors 0 to < 4, Priors 4 to < 7, Priors 7+, ..."
SEN Provision,"[Total EHC plans and statements of SEN, Total ..."
Sex,"[Female, Male]"


In [11]:
chars_df = chars_df[["time_period", "region_name", "geographic_level", "characteristic_type", "characteristic_value",
    "number_of_students_level3", "number_of_students_alev", "number_of_students_acad", "number_of_students_agen",
    "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate",
    "number_of_students_potential", "pc_achieving_3_astar_to_a_alev", "pc_achieving_atleast_two_alev", "aps_per_entry_grade_alev"]]

In [12]:
chars_df = chars_df[(chars_df["geographic_level"] == "Regional") &
                    (chars_df["characteristic_type"].isin(["Disadvantage", "Ethnicity", "Sex", "All Students"]))].reset_index(drop=True)
# 9 regions *5 time periods  12* characteristics = 540 rows

chars_df = chars_df.map(set_nulls)

chars_df["time_period"] = chars_df["time_period"].apply(adjust_time_periods)

In [13]:
chars_df["pc_achieving_alev"] = ((chars_df["number_of_students_alev"] / chars_df["number_of_students_potential"]) *100).round(3)

chars_df.drop(columns=["geographic_level", "number_of_students_level3", "number_of_students_acad", "number_of_students_agen",
                      "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate"],
                      inplace=True)

In [14]:
chars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   time_period                     540 non-null    object 
 1   region_name                     540 non-null    object 
 2   characteristic_type             540 non-null    object 
 3   characteristic_value            540 non-null    object 
 4   number_of_students_alev         540 non-null    int64  
 5   number_of_students_potential    540 non-null    int64  
 6   pc_achieving_3_astar_to_a_alev  540 non-null    float64
 7   pc_achieving_atleast_two_alev   540 non-null    float64
 8   aps_per_entry_grade_alev        540 non-null    object 
 9   pc_achieving_alev               540 non-null    float64
dtypes: float64(3), int64(2), object(5)
memory usage: 42.3+ KB


In [15]:
with open(chars_path, "w") as f:
    chars_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out retention_df

</h3>

    I will:
    - pull the important/ useful columns
    - deal with the custom null values "z", "x" and "c"
    - identify actual null values
    
</div>

In [16]:
retention_df["exam_cohort"].unique()

array(['A level', 'Academic', 'Applied general', 'Tech level',
       'Technical certificate'], dtype=object)

In [17]:
retention_df = retention_df[["time_period", "region_name", "geographic_level", "exam_cohort", "student_count_year_1",
    "student_count_year_2", "retained", "retained_and_assessed", "returned_and_retained", "perc_retained",
    "perc_retained_and_assessed", "perc_returned_and_retained"]]

In [18]:
retention_df = retention_df[retention_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *5 cohorts *5 time period = 225 rows

retention_df = retention_df.map(set_nulls)

retention_df["time_period"] = retention_df["time_period"].apply(adjust_time_periods)

In [19]:
retention_df.drop(columns=["geographic_level"], inplace=True)

In [20]:
retention_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   time_period                 225 non-null    object 
 1   region_name                 225 non-null    object 
 2   exam_cohort                 225 non-null    object 
 3   student_count_year_1        225 non-null    int64  
 4   student_count_year_2        180 non-null    float64
 5   retained                    225 non-null    int64  
 6   retained_and_assessed       225 non-null    int64  
 7   returned_and_retained       180 non-null    float64
 8   perc_retained               225 non-null    float64
 9   perc_retained_and_assessed  225 non-null    float64
 10  perc_returned_and_retained  180 non-null    float64
dtypes: float64(5), int64(3), object(3)
memory usage: 19.5+ KB


In [21]:
with open(retention_path, "w") as f:
    retention_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out results_df

</h3>

    I will:
    - pull the important/ useful columns
    - deal with the custom null values "z", "x" and "c"
    - identify actual null values
    - give ordered categorical type to grade columns in order to perform numerical aggregation later (eg. min, max etc.)

    
</div>

In [22]:
results_df

,time_period,time_identifier,geographic_level,country_code,country_name,version,region_code,region_name,old_la_code,new_la_code,...,perc_a_grade_achieved,perc_b_grade_achieved,perc_c_grade_achieved,perc_d_grade_achieved,perc_e_grade_achieved,perc_astar_a_grade_achieved,perc_astar_b_grade_achieved,perc_astar_c_grade_achieved,perc_astar_d_grade_achieved,perc_astar_e_grade_achieved
0,202223,Academic year,Local authority,E92000001,England,Revised,E12000007,London,318.0,E09000027,...,z,z,z,z,z,z,z,z,z,z
1,202223,Academic year,Local authority,E92000001,England,Revised,E12000007,London,317.0,E09000026,...,z,z,z,z,z,z,z,z,z,z
2,202223,Academic year,Local authority,E92000001,England,Revised,E12000007,London,204.0,E09000012,...,z,z,z,z,z,z,z,z,z,z
3,202223,Academic year,Local authority,E92000001,England,Revised,E12000007,London,205.0,E09000013,...,z,z,z,z,z,z,z,z,z,z
4,202223,Academic year,Local authority,E92000001,England,Revised,E12000007,London,210.0,E09000028,...,z,z,z,z,z,z,z,z,z,z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175703,202324,Academic year,Regional,E92000001,England,Revised,E12000007,London,NaN,NaN,...,27.42067,16.11066,14.07648,16.19203,10.0895,27.42067,43.53133,57.60781,73.79984,83.88934
175704,202324,Academic year,Regional,E92000001,England,Revised,E12000007,London,NaN,NaN,...,z,z,z,z,z,z,z,z,z,z
175705,202324,Academic year,Regional,E92000001,England,Revised,E12000007,London,NaN,NaN,...,16.41791,34.32836,29.85075,8.95522,4.47761,16.41791,50.74627,80.59701,89.55224,94.02985
175706,202324,Academic year,Regional,E92000001,England,Revised,E12000007,London,NaN,NaN,...,9.67742,3.22581,14.51613,12.90323,33.87097,9.67742,12.90323,27.41935,40.32258,74.19355


In [23]:
results_df.groupby(["qualification", "subject_area"]).agg({"subject_name": ["nunique", "unique"]})

subject_name  \
                                             nunique   
qualification subject_area                             
A level       Accounting and Finance               1   
              All STEM subjects                    1   
              All facilitating subjects            1   
              All subjects                         1   
              Anthropology                         1   
...                                              ...   
AS level      Science - Biology                    1   
              Science - Chemistry                  1   
              Science - Physics                    1   
              Science - other                      6   
              Sociology                            1   

                                                                                            
                                                                                    unique  
qualification subject_area                                                                  
A level       Accounting and Finance                        [Total Accounting and finance]  
              All STEM subjects                                      [Total STEM subjects]  
              All facilitating subjects                      [Total facilitating subjects]  
              All subjects                                                [Total subjects]  
              Anthropology                                            [Total Anthropology]  
...                                                                                    ...  
AS level      Science - Biology                                            [Total Biology]  
              Science - Chemistry                                        [Total Chemistry]  
              Science - Physics                                            [Total Physics]  
              Science - other            [Science in Society, Science SA, Environmental...  
              Sociology                                                  [Total Sociology]  

[74 rows x 2 columns]

In [24]:
results_df = results_df[["time_period", "geographic_level", "region_name", "qualification", "subject_area", 
    "subject_name", "entry_count", "perc_astar_grade_achieved", "perc_astar_a_grade_achieved",
    "perc_astar_b_grade_achieved", "perc_astar_c_grade_achieved", "perc_astar_d_grade_achieved",
    "perc_astar_e_grade_achieved"]]

In [25]:
results_df = results_df[results_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *2 qualifications *5 time period *109 subjects = 9810 rows

results_df = results_df.map(set_nulls)

results_df["time_period"] = results_df["time_period"].apply(adjust_time_periods)

In [26]:
results_df.drop(columns=["geographic_level"], inplace=True)

results_df = results_df[~results_df["entry_count"].isna()].reset_index(drop=True)
# remove discontinued subjects as tey are not useful for analysis

In [27]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8847 entries, 0 to 8846
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   time_period                  8847 non-null   object 
 1   region_name                  8847 non-null   object 
 2   qualification                8847 non-null   object 
 3   subject_area                 8847 non-null   object 
 4   subject_name                 8847 non-null   object 
 5   entry_count                  8847 non-null   float64
 6   perc_astar_grade_achieved    5737 non-null   float64
 7   perc_astar_a_grade_achieved  5737 non-null   float64
 8   perc_astar_b_grade_achieved  5737 non-null   float64
 9   perc_astar_c_grade_achieved  5737 non-null   float64
 10  perc_astar_d_grade_achieved  5737 non-null   float64
 11  perc_astar_e_grade_achieved  5737 non-null   float64
dtypes: float64(7), object(5)
memory usage: 829.5+ KB


In [28]:
with open(results_path, "w") as f:
    results_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out stem_df

</h3>

    I will:
    - pull the important/ useful columns
    - deal with the custom null values "z", "x" and "c"
    - identify actual null values
    
</div>

In [29]:
stem_df.groupby("num_maths_science").agg({"sub_name_comb": "nunique"})

,sub_name_comb
num_maths_science,
0,1
1,7
2,16
3,21
4,16
5,7
6,2
Total Students,1


In [30]:
stem_df = stem_df[["time_period", "geographic_level", "region_name", "characteristic_sex", "sub_name_comb",
    "num_maths_science", "num_entered_comb_and_no_other_matsci", "perc_entered_comb_and_no_other_matsci"]]

In [31]:
stem_df = stem_df[stem_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *3 characteristics *5 time periods *71 subjects = 9585 rows

stem_df = stem_df.map(set_nulls)

stem_df["time_period"] = stem_df["time_period"].apply(adjust_time_periods)

In [32]:
stem_df.drop(columns=["geographic_level"], inplace=True)

stem_df = stem_df[stem_df["num_maths_science"] != "Total Students"].reset_index(drop=True)
# 9 regions *3 sexes (due to Null) *5 time periods *70 subjects = 9450 rows

# Remove rows where sub_name_comb contains "one", "two" etc.
stem_df = stem_df[~stem_df["sub_name_comb"].str.contains("Zero|One|Two|Three|Four|Five|Six", case=False, na=False)].reset_index(drop=True)
# 9 regions *3 sexes (due to Null) *5 time periods *63 subjects = 8505 rows

In [33]:
stem_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8505 entries, 0 to 8504
Data columns (total 7 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   time_period                            8505 non-null   object 
 1   region_name                            8505 non-null   object 
 2   characteristic_sex                     8505 non-null   object 
 3   sub_name_comb                          8505 non-null   object 
 4   num_maths_science                      8505 non-null   object 
 5   num_entered_comb_and_no_other_matsci   8505 non-null   int64  
 6   perc_entered_comb_and_no_other_matsci  8505 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 465.2+ KB


In [34]:
with open(stem_path, "w") as f:
    stem_df.to_csv(f, index=False)